In [1]:
import os
import xml.etree.ElementTree as ET
import uuid
import random

import pandas as pd

In [2]:
rd = random.Random()
rd.seed(42)
gen_uuid = lambda: str(uuid.UUID(int=rd.getrandbits(128), version=4))

def is_img(pth):
    return os.path.splitext(pth)[-1].lower() in [".png", ".jpg", ".jpeg", ".bmp"]

In [3]:
def parse_text(txt):
    text = txt.lower()
    if txt == "true":
        return True
    elif txt == "false":
        return False
    else:
        return str(txt).strip("none")

def split_point(s):
    return s.split(";")[0].split(",")

def process_img_node(node: ET.Element) -> list:
    img_attributes = {
        "id": gen_uuid(),
        "name": node.attrib["name"],
        "width": int(node.attrib["width"]),
        "height": int(node.attrib["height"])
    }

    boxes = []
    positions = []
    for child in node:
        attributes = {}
        for meta in child:
            attributes[meta.attrib["name"]] = parse_text(meta.text)

        obj = {
            "label": child.attrib["label"],
            **attributes
        }
        if child.tag == "points":
            obj["x"], obj["y"] = [float(i) for i in split_point(child.attrib["points"])]
            positions.append(obj)
        else:
            obj["xtl"] = float(child.attrib["xtl"])
            obj["ytl"] = float(child.attrib["ytl"])
            obj["xbr"] = float(child.attrib["xbr"])
            obj["ybr"] = float(child.attrib["ybr"])
            boxes.append(obj)

    objects = {}
    for box in boxes:
        idx = f"{box['ID']}-{box['label']}"
        objects[idx] = box
    for pos in positions:
        idx = f"{pos['ID']}-{pos['label']}"
        if idx in objects:
            objects[idx]["x"] = pos["x"]
            objects[idx]["y"] = pos["y"]
        else:
            objects[idx] = pos
    
    for i in objects.values():
        i["img_id"] = img_attributes["id"]
        i["id"] = gen_uuid()

    return list(objects.values()), img_attributes

def process_annotation_file(pth: str, batch: str = None, folder: str = None) -> list:
    tree = ET.parse(pth)
    root = tree.getroot()
    objects = []
    img_attributes = []
    for child in root:
        if child.tag != "image":
            continue
        tmp_objects, tmp_img_attributes = process_img_node(child)
        objects += tmp_objects
        img_attributes.append(tmp_img_attributes)

    for img_attrib in img_attributes:
        img_attrib["batch"] = batch

    for img_attrib in img_attributes:
        img_attrib["folder"] = folder

    return objects, img_attributes

In [4]:
base_path = "../../../db/object-detection"
batches = sorted([i for i in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, i))])

folders = {}
for batch in batches:
    tmp_pth = os.path.join(base_path, batch)
    folders[batch] = sorted([i for i in os.listdir(tmp_pth) if os.path.isdir(os.path.join(tmp_pth, i))])

In [5]:
imgs = {}
annotations = {}
total_img_count = 0
for batch in batches:
    for folder in folders[batch]:
        tmp_pth = os.path.join(base_path, batch, folder)
        batch_imgs = [i for i in os.listdir(tmp_pth) if is_img(i)]
        batch_annotation = [i for i in os.listdir(tmp_pth) if i.endswith(".xml")][0]
        
        key = (batch, folder)
        imgs[key] = batch_imgs
        annotations[key] = batch_annotation
        total_img_count += len(batch_imgs)

print("total images:", total_img_count)

total images: 1032


In [6]:
objects = []
img_attributes = []

for batch, annot in annotations.items():
    tmp_objects, tmp_img_attributes = process_annotation_file(os.path.join(base_path, *batch, annot), *batch)
    objects += tmp_objects
    img_attributes += tmp_img_attributes

df = pd.DataFrame(objects)
df_imgs = pd.DataFrame(img_attributes)
print(df_imgs.shape)

for (batch, folder), batch_imgs in imgs.items():
    for img in batch_imgs:
        if ((df_imgs["name"] == img) & (df_imgs["batch"] == batch) & (df_imgs["folder"] == folder)).any():
            continue

        df_imgs = pd.concat([df_imgs, pd.DataFrame([{"id": gen_uuid(), "name": img, "batch": batch, "folder": folder}])], ignore_index=True)

(980, 6)


In [7]:
print("total number of images: ", df_imgs.shape[0])

print("total number of images with annotations: ", df["img_id"].unique().shape[0])

print("total number of annotated objects: ", df.shape[0])

object_per_img = df.groupby("img_id").count()["id"].values
print(f"objects per img: min = {object_per_img.min()}. max = {object_per_img.max()}. avg = {object_per_img.mean():.02f}")

print("number of object per class:\n", df["label"].value_counts())

total number of images:  1032
total number of images with annotations:  965
total number of annotated objects:  24400
objects per img: min = 1. max = 87. avg = 25.28
number of object per class:
 label
office_chair               7867
desk                       3385
desk_FULL                  2300
laptop_opened              2060
monitor                    1900
keyboard                    936
plants                      881
cabinet_FULL                782
bag                         646
cabinet                     632
dinning_chair               561
laptop_closed               369
bar_stool                   355
coffee_table                306
meeting_room_table          293
lounge_chair                286
whiteboard                  209
sofa                        201
coffee_table_FULL           196
coat                        160
meeting_room_table_FULL      36
printer                      26
coffee_station                7
sink                          6
Name: count, dtype: int64


In [8]:
tmp_df = df[(df["xtl"].isna()) | (df["x"].isna())]
tmp_df = pd.merge(tmp_df, df_imgs, how="left", left_on="img_id", right_on="id")[["batch", "folder", "name", "label", "ID", "xtl", "ytl", "xbr", "ybr", "x", "y"]]

tmp_df.to_csv(os.path.join(base_path, "no_boxes_or_position.csv"), index=False)

print("images with no boxes or position:", tmp_df["name"].unique().shape[0])

tmp_df = pd.merge(df, df_imgs, how="outer", left_on="img_id", right_on="id")
tmp_df = tmp_df[tmp_df["id_x"].isna()][["batch", "folder","name"]].sort_values(by=["batch", "folder"])
tmp_df.to_csv(os.path.join(base_path, "no_annotations.csv"), index=False)
print("images with no annotations:", tmp_df["name"].unique().shape[0])

df1 = df_imgs.groupby(['name','batch', "folder"]).size().reset_index()[['name','batch', "folder"]]
duplicates = df1[df1.duplicated('name', keep=False)]
duplicate_info = duplicates.groupby('name').agg({
    'folder': lambda x: ', '.join(x),
    'batch': lambda x: ', '.join(x)
}).reset_index()
duplicate_info.to_csv(os.path.join(base_path, "duplicates.csv"), index=False)
print("duplicates:", duplicate_info.shape[0])

images with no boxes or position: 50
images with no annotations: 67
duplicates: 62


In [9]:
df.to_csv(os.path.join(base_path, "annotation.csv"), index=False)
df_imgs.to_csv(os.path.join(base_path, "images.csv"), index=False)